<a href="https://colab.research.google.com/github/mafaiziyas/Wearable-AI-Barbell-Activity-Recognition-and-Rep-Counting-Engine/blob/main/scripts/data_aggregation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


5 barbel excercies (Bench press, Deadlift, Overheadpress, Barbell row, Squat) are tracked using a watch with a gyroscope and an accelormeter. There are 5 participants, each csv contains data obtained from 1 participant performing 1 excercise in a certain category (heavy, medium).

All scattered csv files have 6 columns: "epoch (ms)", "time (01:00)", "elapsed (s)", "x-axis (g)", "y-axis (g)", "z-axis (g)"

* **epoch (ms):** Universal Unix timestamp in milliseconds. Will be used as primary key to merge or sync with other sensor data streams.
* **time (01:00):** Human-readable date and time formatted in a local timezone (UTC+1). Redundant for processing and can be dropped.
* **elapsed (s):** Time passed in seconds since the start of the recording session. Useful for calculating sampling rates or elapsed durations.
* **x-axis (g):** Acceleration along the X-axis in gravitational force units (1g≈9.81 m/s 2 ).
* **y-axis (g):** Acceleration along the Y-axis in g.
* **z-axis (g):** Acceleration along the Z-axis in g.

##Reading files

In [45]:
import pandas as pd
from glob import glob

In [46]:
file_paths = glob("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/*.csv")
file_paths[0:4:1]

['/content/drive/MyDrive/Fitness Tracker/Raw Data Files/A-ohp-heavy_MetaWear_2019-01-14T14.55.42.246_C42732BE255C_Gyroscope_25.000Hz_1.4.4.csv',
 '/content/drive/MyDrive/Fitness Tracker/Raw Data Files/B-ohp-heavy2-rpe7_MetaWear_2019-01-11T16.42.43.398_C42732BE255C_Gyroscope_25.000Hz_1.4.4.csv',
 '/content/drive/MyDrive/Fitness Tracker/Raw Data Files/E-bench-medium_MetaWear_2019-01-18T18.12.13.952_C42732BE255C_Accelerometer_12.500Hz_1.4.4.csv',
 '/content/drive/MyDrive/Fitness Tracker/Raw Data Files/A-squat-medium3-rpe7_MetaWear_2019-01-11T17.19.34.896_C42732BE255C_Accelerometer_12.500Hz_1.4.4.csv']

In [47]:
def read_data (file_paths):
  acceleratometer_df = pd.DataFrame()
  gyroscope_df = pd.DataFrame()
  acc_df = 1
  gyro_df = 1

  for f in file_paths:
    participant = f.split("-")[0].replace("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/", "")
    label = f.split("-")[1]
    category = f.split("-")[2].split("_")[0]

    df = pd.read_csv(f)
    df["participant"] = participant
    df["label"] = label
    df["category"] = category

    if "Accelerometer" in f:
      df["set"]= str(acc_df)
      acc_df += 1
      acceleratometer_df = pd.concat([acceleratometer_df, df])
    elif "Gyroscope" in f:
      df["set"]= str(gyro_df)
      gyro_df += 1
      gyroscope_df = pd.concat([gyroscope_df, df])
  return acceleratometer_df, gyroscope_df

acceleratometer_df, gyroscope_df = read_data (file_paths)

In [48]:
def set_index_to_epoch (df):
  df["epoch (ms)"] = pd.to_datetime(df["epoch (ms)"], unit= "ms")
  df.index = df["epoch (ms)"]
  return df

acceleratometer_df = set_index_to_epoch(acceleratometer_df)
gyroscope_df = set_index_to_epoch(gyroscope_df)

In [49]:
def drop_dt_columns_in_df(df):
  df = df.drop(columns=['epoch (ms)', 'time (01:00)', 'elapsed (s)'])
  return df

gyroscope_df = drop_dt_columns_in_df(gyroscope_df)
acceleratometer_df = drop_dt_columns_in_df(acceleratometer_df)

##Merging Gyroscope data with Accelometer

In [50]:
df_merged = pd.concat([acceleratometer_df.loc[:,['x-axis (g)', 'y-axis (g)', 'z-axis (g)']], gyroscope_df], axis=1) #axis 1 is col wise / horizontal
df = df_merged.copy()
df.head()

,x-axis (g),y-axis (g),z-axis (g),x-axis (deg/s),y-axis (deg/s),z-axis (deg/s),participant,label,category,set
epoch (ms),,,,,,,,,,
2019-01-11 15:08:04.950,NaN,NaN,NaN,-10.671,-1.524,5.976,B,bench,heavy1,50
2019-01-11 15:08:04.990,NaN,NaN,NaN,-8.720,-2.073,3.171,B,bench,heavy1,50
2019-01-11 15:08:05.030,NaN,NaN,NaN,0.488,-3.537,-4.146,B,bench,heavy1,50
2019-01-11 15:08:05.070,NaN,NaN,NaN,0.244,-5.854,3.537,B,bench,heavy1,50
2019-01-11 15:08:05.110,NaN,NaN,NaN,-0.915,0.061,-2.805,B,bench,heavy1,50


In [51]:
df.dropna().shape
#significant amout  of data is dropped when .dropna() called

def NA_percent (df):
  return (df.isna().sum()/df.shape[0])*100
NA_percent(df)

,0
x-axis (g),66.161000
y-axis (g),66.161000
z-axis (g),66.161000
x-axis (deg/s),32.233018
y-axis (deg/s),32.233018
z-axis (deg/s),32.233018
participant,32.233018
label,32.233018
category,32.233018
set,32.233018


###Resampling

Gyroscope typically samples faster and has a higher maximum data rate than the accelerometer. Hence more data in gyroscope than accelerometer. Because your gyroscope and accelerometer sample data at different speeds, resampling is the perfect way to force them to use the exact same time intervals so they match up perfectly.

In [53]:
def downsampling (df):
  sampling = {'x-axis (g)':"mean", 'y-axis (g)':"mean", 'z-axis (g)':"mean", 'x-axis (deg/s)':"mean",'y-axis (deg/s)':"mean", 'z-axis (deg/s)':"mean", 'participant':"last", 'label':"last", 'category':"last", "set":"last"}
  df = df.resample(rule="100ms").apply(sampling) #100ms is selected as its the optimal window to consider before start losing critical movement details.
  df_final = df.dropna()
  return df_final

df_final = downsampling(df)
df_final.shape

(17912, 10)

In [54]:
def renaming(df):
  rename_mapper = {'x-axis (g)':"acc_x (g)", 'y-axis (g)':"acc_y (g)", 'z-axis (g)':"acc_z (g)", 'x-axis (deg/s)':"gyr_x (deg/s)", 'y-axis (deg/s)':"gyr_y (deg/s)", 'z-axis (deg/s)':"gyr_z (deg/s)", 'participant':"participant", 'label':"label", 'category':"category", "set":"set"}
  df_final= df.rename(columns = rename_mapper)
  return df_final

df_final = renaming(df_final)
df_final.head()

,acc_x (g),acc_y (g),acc_z (g),gyr_x (deg/s),gyr_y (deg/s),gyr_z (deg/s),participant,label,category,set
epoch (ms),,,,,,,,,,
2019-01-11 15:08:05.300,0.0135,0.9770,-0.071,-1.524333,3.069333,4.573000,B,bench,heavy1,50
2019-01-11 15:08:05.400,-0.0110,0.9700,-0.086,0.732000,-2.104000,-0.518500,B,bench,heavy1,50
2019-01-11 15:08:05.500,0.0080,0.9710,-0.073,-3.292333,-0.081333,3.963667,B,bench,heavy1,50
2019-01-11 15:08:05.600,0.0120,0.9960,-0.051,-8.597500,-2.774500,-1.006000,B,bench,heavy1,50
2019-01-11 15:08:05.700,-0.0040,0.9595,-0.071,9.999667,1.423000,-1.687000,B,bench,heavy1,50


##Final check

In [55]:
df.info()


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 69677 entries, 2019-01-11 15:08:04.950000 to 2019-01-20 17:35:13.702000
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   x-axis (g)      23578 non-null  float64
 1   y-axis (g)      23578 non-null  float64
 2   z-axis (g)      23578 non-null  float64
 3   x-axis (deg/s)  47218 non-null  float64
 4   y-axis (deg/s)  47218 non-null  float64
 5   z-axis (deg/s)  47218 non-null  float64
 6   participant     47218 non-null  object 
 7   label           47218 non-null  object 
 8   category        47218 non-null  object 
 9   set             47218 non-null  object 
dtypes: float64(6), object(4)
memory usage: 5.8+ MB


In [57]:
df.to_pickle("/content/drive/MyDrive/Fitness Tracker/Intermediate Data Files (post Data_preprocessing)/processed_data_file.pkl")